In [1]:
import pandas as pd
import string
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [2]:
df = pd.read_csv('/kaggle/input/datasets/omar2716/emails-dataset/spam_and_ham_classification.csv')

In [3]:
df

,label,text
0,ham,into the kingdom of god and those that are ent...
1,spam,there was flow at hpl meter 1505 on april firs...
2,ham,take a look at this one campaign for bvyhprice...
3,spam,somu wrote actually thats what i was looking f...
4,spam,fathi boudra wrote i fixed the issue in the sv...
...,...,...
9984,ham,this would be a great tragedy for all concerne...
9985,ham,"hello , welcome to medzonline filamentous shop..."
9986,ham,this is amazing stuff add some inches fast saf...
9987,spam,author jra date escapenumber escapenumber esca...


In [4]:
df.head()

,label,text
0,ham,into the kingdom of god and those that are ent...
1,spam,there was flow at hpl meter 1505 on april firs...
2,ham,take a look at this one campaign for bvyhprice...
3,spam,somu wrote actually thats what i was looking f...
4,spam,fathi boudra wrote i fixed the issue in the sv...


In [5]:
df.tail()

,label,text
9984,ham,this would be a great tragedy for all concerne...
9985,ham,"hello , welcome to medzonline filamentous shop..."
9986,ham,this is amazing stuff add some inches fast saf...
9987,spam,author jra date escapenumber escapenumber esca...
9988,ham,anatrim escapenumber the newest and most attra...


In [6]:
df.shape

(9989, 2)

In [7]:
df.size

19978

In [8]:
df.sample(20)

,label,text
9339,ham,girls lie when they say size doesn't matter th...
5013,ham,want to have better sex you dont need viagra y...
7394,spam,the triplot function in the teachingdemos pack...
8280,ham,dear valued member with this special pharmaceu...
619,spam,on thu escapenumber escapenumber escapenumber ...
1721,ham,hi i am bored this evening i am escapenumber y...
537,ham,i just wanted to write and thank you for spur ...
3042,ham,buy x? . n?xmg 30 t?blets for only $ 119 . 95 ...
7208,ham,i strongly recommend you to switch to the cana...
3323,spam,the california litigation team weekly conferen...


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9989 entries, 0 to 9988
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   9989 non-null   object
 1   text    9989 non-null   object
dtypes: object(2)
memory usage: 156.2+ KB


In [10]:
all_words = ' '.join(df.text).split()

In [11]:
len(all_words)

2747483

In [12]:
count = Counter(all_words)
count

Counter({'into': 2295,
         'the': 85264,
         'kingdom': 49,
         'of': 40427,
         'god': 275,
         'and': 42360,
         'those': 1104,
         'that': 17078,
         'are': 10055,
         'entering': 58,
         'in': 28900,
         'he': 3506,
         'lord': 81,
         'pardon': 19,
         'escapenumber': 133208,
         'us': 3213,
         'this': 16485,
         'thing': 580,
         'we': 9072,
         'pray': 32,
         'thee': 22,
         'have': 10014,
         'excused': 1,
         'escapenumbernot': 3,
         'therefore': 310,
         'o': 1720,
         'believers': 14,
         'to': 56587,
         'look': 1183,
         'grave': 25,
         'for': 22671,
         'you': 22222,
         'it': 13230,
         'holy': 52,
         'ghost': 24,
         'being': 1141,
         'a': 36859,
         'habitation': 3,
         'unto': 39,
         'forth': 87,
         'words': 272,
         'truth': 101,
         'soberness': 2,
   

In [13]:
df.label.info()

<class 'pandas.core.series.Series'>
RangeIndex: 9989 entries, 0 to 9988
Series name: label
Non-Null Count  Dtype 
--------------  ----- 
9989 non-null   object
dtypes: object(1)
memory usage: 78.2+ KB


In [14]:
df.describe()

,label,text
count,9989,9989
unique,2,9989
top,ham,anatrim escapenumber the newest and most attra...
freq,5294,1


In [15]:
df.label.value_counts()

label
ham     5294
spam    4695
Name: count, dtype: int64

> Cleaning Data

In [16]:
df.isna().sum()

label    0
text     0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df = df.dropna()
all_words = ' '.join(df.text).split()

In [19]:
df = df.drop_duplicates()
df = df.dropna()

# حذف علامات الترقيم

In [20]:
punc = string.punctuation

In [21]:
punTable = str.maketrans('','',punc)

In [22]:
df.text = df.text.str.translate(punTable)

In [23]:
df

,label,text
0,ham,into the kingdom of god and those that are ent...
1,spam,there was flow at hpl meter 1505 on april firs...
2,ham,take a look at this one campaign for bvyhprice...
3,spam,somu wrote actually thats what i was looking f...
4,spam,fathi boudra wrote i fixed the issue in the sv...
...,...,...
9984,ham,this would be a great tragedy for all concerne...
9985,ham,hello welcome to medzonline filamentous shop\...
9986,ham,this is amazing stuff add some inches fast saf...
9987,spam,author jra date escapenumber escapenumber esca...


# حروف صغيرة

In [24]:
df.text = df.text.str.lower()

# حذف الكلمات الشائعة

In [25]:
# nltk.download('stopwords')
# nltk.download('punkt')
stopWords = set(stopwords.words('english'))

In [26]:
df.text = df.text.str.split()

In [27]:
df.text = df.text.apply(lambda x:[word for word in x if word not in stopWords])

In [28]:
df.text= df.text.apply(lambda x:' '.join([word for word in x if len(word) > 2]))

In [29]:
df

,label,text
0,ham,kingdom god entering lord pardon escapenumber ...
1,spam,flow hpl meter 1505 april first deal ticket de...
2,ham,take look one campaign bvyhprice escapenumber ...
3,spam,somu wrote actually thats looking user entered...
4,spam,fathi boudra wrote fixed issue svn repo rev es...
...,...,...
9984,ham,would great tragedy concerned situation digiti...
9985,ham,hello welcome medzonline filamentous shop plea...
9986,ham,amazing stuff add inches fast safe effective s...
9987,spam,author jra date escapenumber escapenumber esca...


# تحويل الكلمات إلى جذورها الأساسية أو صيغتها المعيارية

In [30]:
# nlp = spacy.load("en_core_web_sm")
# df['text'] = df['text'].apply(lambda x: " ".join([token.lemma_ for token in nlp(str(x))]))

In [31]:
df

,label,text
0,ham,kingdom god entering lord pardon escapenumber ...
1,spam,flow hpl meter 1505 april first deal ticket de...
2,ham,take look one campaign bvyhprice escapenumber ...
3,spam,somu wrote actually thats looking user entered...
4,spam,fathi boudra wrote fixed issue svn repo rev es...
...,...,...
9984,ham,would great tragedy concerned situation digiti...
9985,ham,hello welcome medzonline filamentous shop plea...
9986,ham,amazing stuff add inches fast safe effective s...
9987,spam,author jra date escapenumber escapenumber esca...


# تقسيم البيانات

In [32]:
x = df.text
y = df.label

In [33]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

# هندسة الميزات

In [34]:
tfidf=TfidfVectorizer()

In [35]:
x_train_vec = tfidf.fit_transform(x_train)
x_test_vec = tfidf.transform(x_test)

# بناء النموذج

In [36]:
model = MultinomialNB()

In [37]:
model.fit(x_train_vec,y_train)

MultinomialNB()

In [38]:
pred = model.predict(x_test_vec)

In [39]:
acc = accuracy_score(pred,y_test)

In [40]:
acc

0.970970970970971

In [41]:
classification =classification_report(pred,y_test)

In [42]:
print(classification)

              precision    recall  f1-score   support

         ham       0.96      0.98      0.97      1035
        spam       0.98      0.96      0.97       963

    accuracy                           0.97      1998
   macro avg       0.97      0.97      0.97      1998
weighted avg       0.97      0.97      0.97      1998



In [43]:
import joblib

model_path = (
    "/kaggle/input/models/omar2716/emails-model/scikitlearn/default/1/spam"
)
loaded_model = joblib.load(model_path)
print("تم تحميل النموذج بنجاح:", type(loaded_model))

تم تحميل النموذج بنجاح: <class 'sklearn.pipeline.Pipeline'>


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.2.2 when using version 1.6.1. This might lead to breaking code or in

In [44]:
joblib.dump(model, "spam.joblib")

['spam.joblib']

In [45]:
df.to_csv('output_spam.csv', index=False)

In [46]:
joblib.dump(model, 'spam_model.pkl')

['spam_model.pkl']